In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
 
# 目标函数：Rosenbrock函数
# 计算适应度
def rosenbrock(x):
 
    return 100 * (x[1] - x[0]**2)**2 + (1 - x[0])**2
 
# 初始化种群
# 生成一群鲸鱼，lb坐标数值下限，ub坐标数值上限，pop_size鲸鱼数量，dim鲸鱼的坐标维度，示例中为二维坐标
def initialize_population(pop_size, dim, lb, ub):
 
    return np.random.uniform(lb, ub, (pop_size, dim))
 
# 鲸鱼优化算法
# 鲸鱼优化算法主体,fitness_func计算适应度的函数，max_iter迭代次数,lb, ub, dim, pop_size同前一函数
# 局部变量pop[i]代表目标函数的潜在解，通过在迭代中不断更新潜在解，向最优解逼近，最终得出群组范围内目标函数的满意的解
def whale_optimization_algorithm(fitness_func, lb, ub, dim, pop_size=30, max_iter=1000):
 
    # 初始化种群
 
    pop = initialize_population(pop_size, dim, lb, ub)
 
    # 计算适应度
 
    fitness = np.apply_along_axis(fitness_func, 1, pop)
 
    # 找到最优解
 
    best_idx = np.argmin(fitness) # 最适应的鲸鱼序号
 
    best_whale = pop[best_idx] # 最适应的鲸鱼
 
    best_fitness = fitness[best_idx] # 最适应的鲸鱼的适应度
 
    convergence_curve = []
 
    # 迭代过程
 
    for t in range(max_iter): # 开始迭代
 
        a = 2 - t * (2 / max_iter)  # 线性减少a
 
        for i in range(pop_size): # 遍历鲸鱼群
 
            r1, r2 = np.random.rand(2) # 0到1之间的随机数
 
            A = 2 * a * r1 - a # 收缩包围系数
 
            C = 2 * r2 # 随机系数
 
            p = np.random.rand() # 包围猎物/气泡网捕食的概率
 
            
 
            if p < 0.5: # 50%概率包围猎物
 
                if abs(A) >= 1: # 随机搜索
 
                    # 随机选择一个鲸鱼个体
 
                    random_idx = np.random.randint(0, pop_size) # 随机鲸鱼序号
 
                    X_rand = pop[random_idx] # 随机鲸鱼坐标
 
                    D_X_rand = abs(C * X_rand - pop[i]) # 当前鲸鱼与随机鲸鱼之间的距离
 
                    pop[i] = X_rand - A * D_X_rand # 当前鲸鱼向随机鲸鱼移动
 
                else: #选择最优
 
                    D_best = abs(C * best_whale - pop[i]) # 当前鲸鱼与最优鲸鱼之间的距离
 
                    pop[i] = best_whale - A * D_best # 当前鲸鱼向最优鲸鱼移动
 
            else: # 50%概率气泡网捕食
 
                # 螺旋更新位置
 
                l = (a - 1) * np.exp(-1 * t / max_iter) + 1 # 螺旋形状控制参数
 
                b = 1 # 同为螺旋形状控制参数
 
                pop[i] = best_whale + (abs(pop[i] - best_whale) * np.exp(b * l) * np.cos(2 * np.pi * l)) # 当前鲸鱼通过螺旋上升的方式逐步逼近最优鲸鱼
 
            
 
            # 边界检查
 
            pop[i] = np.clip(pop[i], lb, ub)  # 把位置限制在 [lb, ub] 之间
 
            
 
            # 计算新的适应度
 
            new_fitness = fitness_func(pop[i])
 
            # 更新最优解
 
            if new_fitness < best_fitness:
 
                best_fitness = new_fitness
 
                best_whale = pop[i]
                convergence_curve.append(best_fitness)


 
    return best_whale, best_fitness, convergence_curve
 
# 参数设置
 
pop_size = 30
 
dim = 2
 
lb = -5
 
ub = 5
 
max_iter = 100
 
# 运行WOA
 
best_solution, best_solution_fitness,convergence_curve= whale_optimization_algorithm(rosenbrock, lb, ub, dim, pop_size, max_iter)
print("Best solution: ", best_solution)
print("Best solution fitness: ", best_solution_fitness)

In [ ]:
# ===================== 可视化 =====================

# 1. 收敛曲线
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(convergence_curve, 'b-', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Fitness')
plt.title('Convergence Curve')
plt.grid(True, alpha=0.3)

# 2. 等高线图
x = np.linspace(lb, ub, 400)
y = np.linspace(lb, ub, 400)
X, Y = np.meshgrid(x, y)
Z = 100 * (Y - X**2)**2 + (1 - X)**2

plt.subplot(1, 2, 2)
contour = plt.contourf(X, Y, Z, levels=50, cmap='jet', alpha=0.7)
plt.colorbar(contour)
plt.scatter(best_solution[0], best_solution[1], c='red', s=150, marker='*')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Rosenbrock Contour & Best Solution')

plt.tight_layout()
plt.savefig('plot1.png')
plt.close()

# 3. 3D 图
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='jet', alpha=0.8)
ax.scatter(best_solution[0], best_solution[1], best_solution_fitness, c='red', s=200, marker='*')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('f(x1,x2)')
ax.set_title('Rosenbrock 3D Surface')
plt.colorbar(surf, shrink=0.5, aspect=10)
plt.savefig('plot2.png')
plt.close()

print("✅ Images saved: plot1.png, plot2.png")